### Observables
Wir nennen ein Python Objekt **observable**, falls es folgende Methoden zur Verfügung stellt: 
- `observe(fun: function)`
- `notify(event: str, **kwargs: Any)`  

Die Methode`observe(fun)` registriert die Funktion `fun` unter ihrem Namen als Callback.
Sie nimmt das Schlüssel-Wert Paar `fun.__name__`, `fun` in den Dict `callbacks` auf.
Die Methode `notify(event, **kwargs)` ruft alle registrierten Callbacks `fun` mit den
Argumenten `fun(event, data)` auf.



Ein **Observable**, das in irgendeiner Form dargestellt werden möchte, kann z.B.
nach relevanten Zustandsänderungen jeweils `notify('push_stone', old=(1, 1), new=(2, 3))`
aufrufen. Ein (beoachtendes) Objekt, welches  verantwortlich ist für die graphische Darstellung des Zustands des Observables, kann dann 
ein Callback registrieren, welches aufgrund der Informationen `push_stone` und den Keyword Argumenten 
`old=(1, 1)`, `new=(2, 3)` die Darstellung updaten kann.
Oft hat auch das beoachtende Objekt Zugriff auf das Observable. Dann genügt u.U. schon die
Informationen `event` um Darstellung updaten zu können, jedoch kann zusätzliche Information die Aufgabe erleichtern.

In [ ]:
class Observable:
    def __init__(self):
        self.callbacks = {}

    def observe(self, fun):
        if fun.__name__ not in self.callbacks:
            self.callbacks[f.__name__] = fun

    def unobserve(self, fun):
        if fun.__name__ in self.callbacks:
            self.callbacks.pop(fun.__name__) 

    def notify(self, event, **kwargs):
        for f in self.callbacks.values():
            f(event, **kwargs)

In [ ]:
class Game:
    def __init__(self):
        self.callbacks = []
        self.grid_size = 10
        self.games_played = 0
        self.stones = set()

    def observe(self, fun):
        if fun.__name__ not in self.callbacks:
            self.callbacks[f.__name__] = fun

    def unobserve(self, fun):
        if fun.__name__ in self.callbacks:
            self.callbacks.pop(fun.__name__) 

    def notify(self, event, **kwargs):
        for f in self.callbacks.values():
            f(event, **kwargs)
   

    def new_game(self):
        self.games_played += 1
        self.stones.clear()
        self.notify('newgame', games_played=self.games_played)

    def is_inside(self, pos):
        return (0 <= pos[0] < self.grid_size
                and 0 <= pos[1] < self.grid_size)

    def place_stone(self, pos):
        if not self.is_inside(pos) or pos in self.stones:
            self.notify('error',
                        msg=f'playing at Positon {pos} is illegal',
                        )
            return

        self.stones.add(pos)
        self.notify('place_stone', pos=pos)

    def move_stone(self, old, new):
        if new in self.stones or old not in self.stones:
            self.notify('error',
                        msg=f'cannot move from {old} to {new}!',
                        )
            return

        self.stones.remove(old)
        self.stones.add(new)
        self.notify('move_stone', old=old, new=new)

    def push_stone(self, pos, dpos):
         if new in self.stones or old not in self.stones:
            self.notify('error',
                        msg=f'cannot move from {old} to {new}!',
                        )
            return
        
        dx, dy = dpos
        

    def __repr__(self):
        return f'Games played: {self.games_played}, grid size: {self.grid_size},\n'\
               f'stones: {self.stones}'

In [ ]:
game = Game()
game

In [ ]:
old = (1, 2)
game.place_stone(old)
game

In [ ]:
new = (3, 3)
game.move_stone(old, new)
game

In [259]:
class Game:
    def __init__(self):
        self.callbacks = {}
        self.grid_size = 8
        self.games_played = 0
        self.pos = (4, 4)

    
    def observe(self, fun):
        self.callbacks[fun.__name__] = fun

    def unobserve(self, fun):
        if fun.__name__ in self.callbacks:
            self.callbacks.pop(fun.__name__) 

    def notify(self, event, **kwargs):
        for f in self.callbacks.values():
            f(event, **kwargs)
    

    def new_game(self):
        self.games_played += 1
        self.pos = (4, 4)
        self.notify('new_game', games_played=self.games_played)

    def is_inside(self, pos):
        return (0 <= pos[0] < self.grid_size
                and 0 <= pos[1] < self.grid_size)

    def push_stone(self, dpos):
        dx, dy = dpos
        x, y = self.pos
        npos = (x+dx, y + dy)
        
        if not self.is_inside(npos):
            self.notify('error',
                        msg=f'cannot push stone from  {self.pos} to {npos}!',
                        )
            return
        
        self.pos = npos
        self.notify('push_stone', old=(x, y), new=npos)
        

    def __repr__(self):
        return f'Games played: {self.games_played}, grid size: {self.grid_size},\n'\
               f'pos: {self.pos}'

In [260]:
game = Game()
game

Games played: 0, grid size: 8,
pos: (4, 4)

In [261]:
game.push_stone((1, 2))
game

Games played: 0, grid size: 8,
pos: (5, 6)

In [262]:
def report(event, **kwargs):
    print(event, kwargs)

In [263]:
game.observe(report)

In [264]:
game.new_game()
game

new_game {'games_played': 1}


Games played: 1, grid size: 8,
pos: (4, 4)

In [265]:
game.push_stone((6, 6))
game

error {'msg': 'cannot push stone from  (4, 4) to (10, 10)!'}


Games played: 1, grid size: 8,
pos: (4, 4)

In [266]:
game.push_stone((2, 3))
game

push_stone {'old': (4, 4), 'new': (6, 7)}


Games played: 1, grid size: 8,
pos: (6, 7)

In [269]:
import canvas_helpers as H
from ipycanvas import Canvas
from ipywidgets import Output
from IPython.display import display


err_out = Output(layout={'border': '1px solid black'})


class C:
    def __init__(self, game):
        self.game = game
        self.canvas = Canvas(width=20, height=20, layout={'border': '1px solid black'})
        self.canvas.on_key_down(self.on_key_down)
 
    @err_out.capture(clear_output=True)
    def on_key_down(self, key, *flags):
        key_direction = {'ArrowUp': (0, -1),
                         'ArrowDown': (0, 1),
                         'ArrowRight': (1, 0),
                         'ArrowLeft': (-1, 0),
                         }
        if key in key_direction:
            dpos = key_direction[key]
            game.push_stone(dpos)
        elif key == 'n':
            game.new_game()

    def _ipython_display_(self):
        # display(self.canvas, err_out)
        display(self.canvas)
        self.canvas.focus()

In [270]:
c = C(game)
c

Canvas(height=20, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='…

In [271]:
import canvas_helpers as H
from ipycanvas import MultiCanvas
from ipywidgets import Output
from IPython.display import display


class V:
    def __init__(self, game, boardspec=(20, 20, 20, 20, 8, 8)):
        self.game = game
        self.boardspec = boardspec
        
        self.out = Output(layout={'border': '1px solid black'})
        self.mcanvas = MultiCanvas(2, width=200, height=200, layout={'border': '1px solid black'})
        self.bg, self.fg = self.mcanvas
        H.draw_grid(self.fg, boardspec, line_width=3, color='blue')
        
        self.game.observe(self.update)
        self.game.new_game()

    def new_game(self, **kwargs):
        self.bg.clear()
        H.fill_field(self.bg, self.game.pos, self.boardspec, color='red')
        self.out.append_stdout(f'Round {kwargs['games_played']} started at {self.game.pos}\n')
        
    def push_stone(self, **kwargs):
        H.clear_field(self.bg, kwargs['old'], self.boardspec)
        H.fill_field(self.bg, kwargs['new'], self.boardspec, color='red')
        
    def error(self, **kwargs):
        # with self.out:
        #     print(f'ERROR: {kwargs['msg']}')
        self.out.append_stdout(f'ERROR: {kwargs['msg']}\n')

    def update(self, event,  **kwargs):
        # self.out.append_stdout(f'{event}, {kwargs}\n')
        if hasattr(self, event):
            getattr(self, event)(**kwargs)

    def _ipython_display_(self):
        display(self.mcanvas, self.out)

In [272]:
view = V(game)
#display(c, view)

new_game {'games_played': 2}


In [256]:
err_out.clear_output()

In [257]:
err_out

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

In [258]:
view = V(game)
display(c, view)

Canvas(height=20, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='…

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

MultiCanvas(height=200, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

In [215]:
game.callbacks

{}

In [229]:
with view.out:
    print('foo')

In [210]:
game.callbacks['update']('error', msg='test')

In [188]:
game.unobserve(report)

In [ ]:
 H.fill_field(view.bg, game.pos, view.boardspec, color='red')

In [ ]:
game.callbacks

In [ ]:
game.new_game()
game

In [ ]:
view.bg.clear()

In [ ]:
import canvas_helpers as H
from ipycanvas import Canvas
from ipywidgets import Output
from IPython.display import display


err_out = Output(layout={'border': '1px solid black'})


class CV:
    def __init__(self, game, boardspec=(20, 20, 20, 20, 8, 8)):
        self.game = game
        self.boardspec = boardspec

        self.canvas = Canvas(width=200, height=200, layout={'border': '1px solid black'})
        self.canvas.on_mouse_down(self.on_mouse_down)
        self.canvas.on_mouse_up(self.on_mouse_up)

        self.out = Output(layout={'border': '1px solid black'})
        H.draw_grid(self.canvas, boardspec, line_width=2, color='blue')

        self.click_pos = None

    @err_out.capture(clear_output=True)
    def on_mouse_down(self, x, y):
        self.click_pos = H.xy2cr(x, y, self.boardspec)

    @err_out.capture(clear_output=True)
    def on_mouse_up(self, x, y):
        release_pos = H.xy2cr(x, y, self.boardspec)
        if self.click_pos != release_pos:
            self.game.move_stone(self.click_pos, release_pos)
        else:
            self.game.place_stone(release_pos)
        self.click_pos = None

    def _ipython_display_(self):
        display(self.canvas, self.out, err_out)

In [ ]:
view = CV(game)
view

In [ ]:
view.out.clear_output()

In [ ]:
@view.out.capture()
def report(event, **kwargs):
    print(event, kwargs)

In [ ]:
report.__name__

In [ ]:
game.observe(report)

In [ ]:
game.new_game()
game.place_stone((1, 1))
game.move_stone((1, 1), (1, 1))

In [ ]:
game

In [ ]:
def draw(event, **kwargs):
    if event == 'place_stone':
        H.fill_field(view.canvas, kwargs['pos'], view.boardspec, color='red')
    if event == 'move_stone':
        H.clear_field(canvas, kwargs['pos'], view.board_spec)
        H.fill_field(view.canvas, kwargs['pos'], view.boardspec, color='red')

In [ ]:
game.observe(draw)

In [ ]:
import canvas_helpers as H
from ipycanvas import Canvas
from ipywidgets import Output
from IPython.display import display


err_out = Output(layout={'border': '1px solid black'})


class CV1:
    def __init__(self, game, boardspec=(20, 20, 20, 20, 8, 8)):
        self.game = game
        self.boardspec = boardspec

        self.canvas = Canvas(width=20, height=20, layout={'border': '1px solid black'})
        self.canvas.on_key_down(self.on_key_down)
       

        self.out = Output(layout={'border': '1px solid black'})
       

       

    @err_out.capture(clear_output=True)
    def on_key_down(self, key, *flags):
        key_direction = {'ArrowUp': (0, -1),
                         'ArrowDown': (0, 1),
                         'ArrowRight': (1, 0),
                         'ArrowLeft': (-1, 0),
                         }
        if key in key_direction:
            dpos = key_direction[key]
            game.move_stone(dpos)

    def _ipython_display_(self):
        display(self.canvas, self.out, err_out)

***
Nachstehende Klasse `GridPoint` verwaltet einen Gitterpunkt, dessen Position durch ein
Tuple (row, col) spezifiziert ist und im Attribut `pos` abgelegt ist.
Die Methode `move` &auml;ndert die Position des Gitterpunktes, `reset` setzt sie auf `(0,0)`.

Die Methoden  `move` und `reset` rufen an passender Stelle
`_notify('reset', initial_pos)` und `_notify('move', (old_pos, new_pos))` auf.

Die Argumente von `_notify` tragen jeweils genug Information, um die Position updaten zu k&ouml;nnen. `old_pos` ist nicht unbedingt n&ouml;tig, erleichtert aber die
Aufgabe f&uuml;r die weiter unten definierte Klasse `View`, welche die Position des
Gitterpunkts in  einem Gitter auf der Leinwand darstellt. Es kann so das Gitterfeld mit dem alten Punkt gel&ouml;scht werden.


Der Benutzer der Klasse `GridPoint` soll die Methode `_notify` nicht selber, z.B. via 
`gp._notify` aufrufen (wo `gp` Instanz von `GridPoint`).
Deshalb beginnt `_notify` mit einem `_`, was signalisiert, dass dies eine **private** Methode ist.
***

In [ ]:
class GridPoint(Observable):
    
    def __init__(self):
        self.pos =  (0, 0) # row, col
    
    def reset(self):
        self.pos = (0, 0) 
        self._notify('reset', self.pos)
        
    def move(self, down, right):
        row, col = self.pos
        new_pos = (row + down, col + right)
        self._notify('move', (self.pos, new_pos))
        self.pos = new_pos
        
    def __repr__(self):
        return 'GridPoint({})'.format(self.pos)

In [ ]:
gp = GridPoint()
gp.register_callback(print)
gp

In [ ]:
gp.move(2, 3)
gp.move(-1, 0)
gp

In [ ]:
gp.reset()
gp

***
Folgende Klasse `View` stellt den Gitterpunkt als Feld eines Gitters dar.
***

In [ ]:
def draw_grid(canvas, n = 10):
    # horizontal lines
    dy = canvas.height/n
    for i in range(n + 1):
        y = i * dy
        canvas.stroke_lines([(0, y), (canvas.width, y)])
        
    # vertical lines
    dx = canvas.width/n
    for i in range(n + 1):    
        x = i * dx
        canvas.stroke_lines([(x, 0), (x, canvas.height)])

In [ ]:
from ipycanvas import MultiCanvas
class View:
    def __init__(self):
        self.canvas = MultiCanvas(2, width = 200, height = 200, layout = {'border': '2px solid black'})
        self.bg, self.fg = self.canvas
        draw_grid(self.fg)
        
    def row_col2xy(self, row, col):
        return (20*col, 180 - 20*row)
    
    def update(self, event, data):
        if event == 'reset':
            self.bg.clear()
            self.bg.fill_rect(*self.row_col2xy(*data), width = 20)
            
        elif event == 'move':
            self.bg.clear_rect(*self.row_col2xy(*data[0]), width = 20)
            self.bg.fill_rect(*self.row_col2xy(*data[1]), width = 20)
        
    def _ipython_display_(self):  
        display(self.canvas)  

In [ ]:
gp = GridPoint()
view = View()
gp.register_callback(view.update)
view

In [ ]:
gp.reset()

In [ ]:
gp.move(3,1)

In [ ]:
gp.move(-3,-1)

***
Oft ist es zweckm&auml;ssig, in der beobachtenden Klasse Methoden  mit den Namen der Events zu definieren. Nachstehend sind das die Methoden `reset` und `move`.  
Die registrierte Callback `update` ruft nun einfach die Methode mit dem Namen des Events auf.
Das Argument sind die mitgelieferten Daten. 
***

In [ ]:
class View:
    def __init__(self):
        self.canvas = MultiCanvas(2, width = 200, height = 200, layout = {'border': '2px solid black'})
        self.bg, self.fg = self.canvas
        draw_grid(self.fg)
        
    def row_col2xy(self, row, col):
        return (20*col, 180 - 20*row)
    
    def reset(self, pos):
        self.bg.clear()
        self.bg.fill_rect(*self.row_col2xy(*pos), width = 20)
    
    def move(self, data):
        self.bg.clear_rect(*self.row_col2xy(*data[0]), width = 20)
        self.bg.fill_rect(*self.row_col2xy(*data[1]), width = 20)
        
    def update(self, event, data):
        getattr(self, event)(data)
        
    def _ipython_display_(self):  
        display(self.canvas)  

In [ ]:
gp = GridPoint()
view = View()
gp.register_callback(view.update)
view

In [ ]:
gp.reset()

In [ ]:
gp.move(2, 1)

***
In nachstehender Variante geben wir der View Zugriff auf das GridPoint-Objekt. 
Dann gen&uuml;gt  die
Informationen `'reset'` oder `'move'`, um die Darstellung updaten zu k&ouml;nnen.
***

In [ ]:
class GridPoint(Observable):
    
    def __init__(self):
        self.pos = self.pos = (0, 0) # row, col
        self.history = []
        
    def reset(self):
        self.pos = (0, 0) # row, col
        self.history.clear()
        self._notify('reset')
        
    def move(self, down, right):
        self.history.append(self.pos)
        row, col = self.pos
        self.pos = (row + down, col + right)
        self._notify('move')
      
    def __repr__(self):
        return 'GridPoint({})'.format(self.pos)
    
class View:
    def __init__(self, gridpoint):
        self.gridpoint = gridpoint
        self.canvas = MultiCanvas(2, width = 200, height = 200, layout = {'border': '2px solid black'})
        self.bg, self.fg = self.canvas
        draw_grid(self.fg)
        
    def row_col2xy(self, row, col):
        return (20*col, 180 - 20*row)
    
    def reset(self, _):
        self.bg.clear()
        x, y = self.row_col2xy(*self.gridpoint.pos)
        self.bg.fill_rect(x, y, width = 20)
    
    def move(self, _):
        x0, y0 = self.row_col2xy(*self.gridpoint.history[-1])
        x1, y1 = self.row_col2xy(*self.gridpoint.pos)
        self.bg.clear_rect(x0, y0, width = 20)
        self.bg.fill_rect(x1, y1, width = 20)
        
    def update(self, event, data):
        getattr(self, event)(data)
        
    def _ipython_display_(self):  
        display(self.canvas)      

In [ ]:
gp = GridPoint()
view = View(gp)
gp.register_callback(view.update)
view

In [ ]:
gp.reset()

In [ ]:
gp.move(3,1)